### 🎯 El Reto
### Título: Análisis Global de Datos Demográficos
**1. El Reto y Origen de los Datos**
En este proyecto, nuestro grupo se enfoca en la extracción y el análisis de datos demográficos globales utilizando la API pública RestCountries. Esta interfaz nos permite acceder a información actualizada y normalizada de más de 250 países y territorios, proporcionando una base sólida para el estudio de patrones geográficos.   
**2. Objetivo Estadístico**    
El núcleo de nuestra investigación consiste en determinar si existe una relación matemática predecible entre la extensión territorial de una nación (Área) y el número de habitantes que alberga (Población).   
Para ello, aplicaremos dos herramientas fundamentales de la estadística descriptiva e inferencial:    
*1. Correlación de Pearson:* Para medir la fuerza y la dirección de la relación entre ambas variables ¿a más territorio, siempre hay más gente?.    
*2. Regresión Lineal Simple:* Para intentar construir un modelo matemático que nos permita predecir la población potencial de un país conociendo únicamente su superficie.   
**3. Justificación**   
Comprender la correlación entre área y población es vital para el análisis de la densidad demográfica.    
Este estudio nos permitirá identificar "outliers" o casos atípicos: países con inmensos territorios pero baja población frente a naciones con superficies mínimas y altísima concentración humana.   

##Celda 2:

### 🗝 Conceptos Clave

**API (Application Programming Interface)** : Un conjunto de reglas que permite que nuestro código se comunique con el servidor de restcountries para obtener datos actualizados sin descargar archivos manuales.

**Deserialización de JSON**: El proceso de convertir el texto plano que envía la API en objetos de Python (diccionarios y listas) que podamos manipular.

**Normalización de Datos**: Técnica para organizar datos complejos (como los nombres de países en varios idiomas) en una estructura de tabla simple y uniforme (DataFrame).



Celda 3 (Código): 
### 🗺 Consumo de la API y DataFrame

In [ ]:
import requests
import pandas as pd
import numpy as np

# Forzamos la URL sin espacios y añadimos un 'User-Agent' 
# Esto engaña a la API para que crea que somos un navegador real

url = "https://restcountries.com/v3.1/all?fields=name,capital,currencies,languages,region,population,area"
response = requests.get(url)

data = response.json()

rows = []

for country in data:
    rows.append({
        "country": country.get("name", {}).get("common"),
        "region": country.get("region"),
        "population": country.get("population"),
        "area": country.get("area"),
         })
    
df = pd.DataFrame(rows)

# Limpieza básica
df = df.dropna(subset=["population", "area"])
df = df[df["area"] > 0]
df ["area"] = df["area"].astype(int)

# Variable derivada útil
df["density"] = (df["population"] / df["area"]).round(2)

df_muestra = df.sample(30, random_state=42).copy()
#df_filtrado = df[df["country"].isin(df_muestra)]

df_filtrado = df[df["country"].isin(paises_elegidos)]

df_muestra.head(30)

# Ordenamos la muestra por la columna 'region'
df_muestra_ordenada = df_muestra.sort_values(by="region")
# Mostramos los 30 países (o los que tenga la muestra)
display(df_muestra_ordenada)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Estilo configurado! 🎨")


Celda 4 (Código):
### Bloque del Grupo (Ejecución)

In [ ]:
import scipy
from scipy import stats

#Calcular correlación
# Datos: área vs población
personas = df_muestra_ordenada ["population"]
area = df_muestra_ordenada ["area"]

# Calcular correlación
r, p_value = stats.pearsonr(personas, area)

print(f"Correlación de Pearson: {r:.2f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
import matplotlib.pyplot as plt

## Visualizar correlación Pearson

plt.figure(figsize=(10, 6))
plt.scatter(personas, area, s=100, alpha=0.6)
plt.xlabel("Personas")
plt.ylabel('Area')
plt.title(f'Correlación: r = {r:.2f}')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns

# Matriz de correlación de Pearson

df = pd.DataFrame({
    'personas': df_muestra_ordenada["population"],
    'area': df_muestra_ordenada["area"],
    'densidad': df_muestra_ordenada["density"]
})
# Calcular todas las correlaciones
corr_matrix = df.corr()
print(corr_matrix, p_value)



# Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1)
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# Spearman

from scipy.stats import spearmanr
result = spearmanr(df[["area", "personas", "densidad" ]])
cols = ["area", "personas", "densidad"]
corr_df = pd.DataFrame(result.statistic, index=cols, columns=cols)
p_df    = pd.DataFrame(result.pvalue,    index=cols, columns=cols)
print("=== Correlacion Spearman ===")
print(corr_df.round(3))
print("\n=== p-values ===")
print(p_df.round(4))

In [ ]:
import matplotlib.pyplot as plt

## Visualizar correlación Spearman

plt.figure(figsize=(10, 6))
plt.scatter(personas, area, s=100, alpha=0.6)
plt.xlabel("Personas")
plt.ylabel('Area')
plt.title(f'Correlación: r = {r:.2f}')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
## Visualizar correlación Spearman

# 3. Gráfico de dispersión (Scatter plot)
plt.figure(figsize=(7,5))
sns.heatmap(corr_df, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Matriz de correlación de Spearman")
plt.show()



Celda 4 (Código):
### Bloque del Grupo (Ejecución)

Celda 5 (Texto):

### 📊 Interpretación del Resultado: 

(INCORPORAREMOS LAS CONCLUSIONES DE NUESTRO ANALISIS)

"Al ejecutar el análisis, observamos ...................................................................................."

Celda 6 (Texto):

### ✂ 2 Errores Frecuentes

1. **Manejo de valores nulos (NaN)**: Algunos países pequeños no tienen informada el área o la población en la API. 

* Cómo evitarlo: Usar el método .get() con valores por defecto o limpiar el DataFrame con df.fillna(0).

2. **Exceso de peticiones (Rate Limiting)**: Hacer requests.get() dentro de un bucle for muy grande. 

* Cómo evitarlo: Hacer una sola petición masiva (como all) y filtrar los datos localmente en Python.

Celda 7 (Texto): 

### 🔗 Conexión con otro Grupo